In [ ]:
from bs4 import BeautifulSoup
from neo4j import GraphDatabase
import os
import json

with open('neo4j_dbinfo', 'r', encoding='utf8') as f:
    neo4j_info = json.load(f)

# === Neo4j Connection ===
URI = neo4j_info["uri"]
AUTH = (neo4j_info["username"], neo4j_info["password"])
driver = GraphDatabase.driver(URI, auth=AUTH)

# ===== PARSING =====
def parse_html(html_content, register_id):
    """Parse product HTML and return structured dict."""
    soup = BeautifulSoup(html_content, "html.parser")
    data = {}

    # product name
    name_tag = soup.find("dt", string="제품명")
    if name_tag:
        data["product_name"] = name_tag.find_next("dd").get_text(strip=True)

    # category
    category_tag = soup.find("dt", string="분류")
    if category_tag:
        data["category"] = category_tag.find_next("dd").get_text(" ", strip=True)

    # usage
    usage_tag = soup.find("dt", string="품목.용도")
    if usage_tag:
        data["usage"] = usage_tag.find_next("dd").get_text(" ", strip=True)

    # packaging
    form_tag = soup.find("dt", string="제품제형")
    if form_tag:
        data["form"] = form_tag.find_next("dd").get_text(strip=True)

    # weight
    weight_tag = soup.find("dt", string="중량·용량·매수·크기")
    if weight_tag:
        data["weight"] = weight_tag.find_next("dd").get_text(" ", strip=True)

    # manufacturer
    manu_tag = soup.find("dt", string="제조국명, 제조회사")
    if manu_tag:
        data["manufacturer"] = manu_tag.find_next("dd").get_text(" ", strip=True)

    # importer
    importer_tag = soup.find("dt", string="수입자,  주소, 연락처")
    if importer_tag:
        data["importer"] = importer_tag.find_next("dd").get_text(" ", strip=True)

    # instructions
    instr_tag = soup.find("dt", string="사용방법")
    if instr_tag:
        data["instructions"] = instr_tag.find_next("dd").get_text(" ", strip=True)

    # emergency
    emerg_tag = soup.find("dt", string="응급처치")
    if emerg_tag:
        data["emergency"] = emerg_tag.find_next("dd").get_text(" ", strip=True)

    # unique ID
    if "product_name" in data:
        data["uid"] = f"{register_id}_{data['product_name']}"

    # chemicals (CAS + name)
    chemicals = []
    for span in soup.select("span.view-item"):
        chem_name = span.get_text(strip=True).rstrip(",")
        button = span.find("button")
        cas = None
        if button and "popMaterialDetail" in button.get("onclick", ""):
            cas = button["onclick"].split("'")[1]
        if cas:
            chemicals.append({"cas": cas, "name": chem_name})
    data["chemicals"] = chemicals

    return data

# ===== NEO4J INSERTION =====
def insert_product(tx, product):
    # Split chemicals into two lists: CAS numbers and names
    # 원본 리스트
    chemicals_cas = [chem["cas"] for chem in product.get("chemicals", []) if "cas" in chem]
    chemicals_name = [chem["name"] for chem in product.get("chemicals", []) if "name" in chem]

    # 중복 제거하면서 두 리스트를 맞춤
    seen = set()
    unique_cas = []
    unique_name = []

    for cas, name in zip(chemicals_cas, chemicals_name):
        if cas not in seen:
            seen.add(cas)
            unique_cas.append(cas)
            unique_name.append(name)    

    # Product 노드 생성
    query = """
    MERGE (p:Product {uid: $uid})
    SET p += $props,
        p.chemicals_cas = $chemicals_cas,
        p.chemicals_name = $chemicals_name
    WITH p
    UNWIND range(0, size($chemicals_cas)-1) AS idx
    MATCH (c:Chemical {casrn: $chemicals_cas[idx]})
    MERGE (p)-[:CONTAINS]->(c)
    """
    props = {k: v for k, v in product.items() if k not in ["uid", "chemicals"]}
    
    tx.run(query, uid=product["uid"], props=props,
           chemicals_cas=unique_cas, chemicals_name=unique_name)


# ===== PROCESS HTML FILE =====
def process_html_file(file_path, register_id):
    with open(file_path, encoding="utf-8") as f:
        html = f.read()
    product = parse_html(html, register_id)
    with driver.session() as session:
        session.write_transaction(insert_product, product)
    print(f"Inserted {product['uid']} with {len(product['chemicals'])} chemicals")


HTML_DIR = "htmls"

# ===== MAIN LOOP =====
for filename in os.listdir(HTML_DIR):
    if filename.endswith(".html"):
        register_id = filename.split("+")[0]  # e.g. FB19-08-0022.html → FB19-08-0022
        process_html_file(os.path.join(HTML_DIR, filename), register_id)